# Structured Output Across Models: Fallbacks, Drift, and Schema Insurance

No matter how good your primary model is, it *will* fail. Sometimes it times out.
Sometimes it hallucinates. Sometimes it produces output that looks syntactically correct
but is semantically wrong. In production, you can't afford to pretend this won't happen.

**Every agent system needs a fallback.** But here's the catch: your overall agent behavior
changes when the model changes. So you need to account for that while you're designing
the system — not after the first outage at 2am. You need to test your system not only
with your primary model, but also with your "second choice."

This notebook demonstrates a **three-tier fallback strategy** and proves why each tier
matters:

1. **Tier 1 — Strict JSON Schema mode**: The provider constrains token generation to match the schema. Best case, but not every model supports it, and it only constrains *shape*, not *judgment*.
2. **Tier 2 — Instructor + Pydantic validators**: We enforce the schema ourselves with validation, normalization, and automatic retries. Works with any model that can produce JSON.
3. **Tier 3 — Prompt-only + canonicalization**: Last resort. We describe the schema in the prompt, hope the model cooperates, then normalize whatever comes back.

We test four models via OpenRouter, each asked to triage the same customer message into a
structured support ticket — and show that even with all three tiers, models disagree on
subjective fields like priority and sentiment.


### 1. Strict JSON schema mode constrains shape, not judgment
All models that supported strict mode produced valid JSON, but they **disagreed on the
values**. Priority `p0` vs `p1` determines whether someone gets paged. Sentiment `angry`
vs `frustrated` changes the escalation path. The schema enforces structure, not
interpretation. And in a real agent with tool calls and multi-turn handoffs, even the
structural guarantee only applies per-call, not across your pipeline.

### 2. Not all models support strict mode you need Tier 2
When your primary model is down and you route to a fallback, that fallback might not
support `response_format=json_schema`. Without Instructor + Pydantic validators as your
second line of defense, your pipeline crashes. Instructor gives you validation *and*
automatic retries with error feedback — the model gets told exactly what went wrong and
gets another chance.

### 3. Pydantic validators are where normalization belongs
Moving the normalization maps *into* the Pydantic model as `field_validator`s means
every path — strict, Instructor, and degraded, benefits from the same normalization.
`"High"` → `"p1"`, `"authentication"` → `"login"`, `"anxious"` → `"frustrated"`. This
isn't optional; it's schema insurance.

### 4. Test your fallback with your entire pipeline
Don't just test "does this model generate text?" Test: does its output survive your
three-tier fallback → parsing → normalization → validation → handoff chain? If any step
fails, the fallback is useless.

### 5. Latency increases as you fall through tiers
Tier 1 (strict) is fastest, one API call, no retries. Tier 2 (Instructor) may need
retries on validation failure. Tier 3 (degraded + canonicalize) requires a separate API
call plus post-processing. Factor this into your SLAs and timeout budgets.

### 6. Different models have different "judgment"
This is subtle but critical: two models looking at the same angry customer may classify
the ticket as `p0/angry` or `p1/frustrated`. If your routing logic, SLA timers, or
escalation rules depend on these fields, **you need to know how each model in your
fallback chain rates them**, and whether that difference is acceptable for your use case.
This is why you test the fallback model as part of the Hardening stage, not after your
first production outage.

In [1]:
!pip install instructor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 22.0 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.13.0
    Uninstalling jiter-0.13.0:
      Successfully uninstalled jiter-0.13.0


In [2]:
import json, re, time, random, requests
from typing import Any, Dict, List, Literal
from pydantic import BaseModel, field_validator
import os
from dotenv import load_dotenv
import instructor
from openai import OpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
BASE = "https://openrouter.ai/api/v1"

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Normalization Maps

These maps are the core of our defense against model drift. Different models will return
different values for the same semantic concept — `"High"` instead of `"p1"`,
`"authentication"` instead of `"login"`, `"anxious"` instead of `"frustrated"`.

We define them early because both the Pydantic validators (used by Instructor in Tier 2)
and the last-resort `canonicalize()` function (Tier 3) depend on them.

In [3]:
CATEGORY_MAP = {
    "authentication": "login",
    "authentication/login": "login",
    "account access": "login",
    "login": "login",
    "billing": "billing",
    "bug": "bug",
    "performance": "performance",
    "feature_request": "feature_request",
    "other": "other",
}

PRIORITY_MAP = {
    "p0": "p0", "p1": "p1", "p2": "p2", "p3": "p3",
    "critical": "p0",
    "urgent": "p1",
    "high": "p1",
    "medium": "p2",
    "low": "p3",
}

SENTIMENT_MAP = {
    "calm": "calm",
    "frustrated": "frustrated",
    "angry": "angry",
    "frustrated/urgent": "frustrated",
    "urgent": "frustrated",
    "anxious": "frustrated",
}

## The Target Schema and Pydantic Model

Two separate things serve two separate purposes here:

1. **The JSON Schema** goes to the API provider for strict mode (Tier 1). The provider
   uses it for constrained decoding at the token level.
2. **The Pydantic model** with `Literal` types and `field_validator`s is our *local*
   enforcement. It normalizes incoming values using the maps above and rejects anything
   that can't be mapped. Instructor (Tier 2) uses this model to validate responses and
   retry on failure.

Note the `field_validator`s: when a model returns `"priority": "High"`, the validator
maps it to `"p1"` via `PRIORITY_MAP`. If the value can't be mapped at all, the validator
raises `ValueError` — and Instructor catches that, injects the error into the prompt,
and gives the model another chance with explicit feedback about what went wrong.

In [4]:
SUPPORT_TICKET_SCHEMA = {
    "type": "object",
    "properties": {
        "ticket_id": {"type": "string"},
        "summary": {"type": "string"},
        "category": {"type": "string", "enum": ["billing","login","bug","feature_request","performance","other"]},
        "priority": {"type": "string", "enum": ["p0","p1","p2","p3"]},
        "customer_sentiment": {"type": "string", "enum": ["calm","frustrated","angry"]},
        "repro_steps": {"type": "array", "items": {"type": "string"}},
        "expected_behavior": {"type": "string"},
        "actual_behavior": {"type": "string"},
        "suggested_next_action": {"type": "string"},
    },
    "required": [
        "ticket_id","summary","category","priority","customer_sentiment",
        "repro_steps","expected_behavior","actual_behavior","suggested_next_action"
    ],
    "additionalProperties": False,
}


class SupportTicket(BaseModel):
    ticket_id: str
    summary: str
    category: Literal["billing", "login", "bug", "feature_request", "performance", "other"]
    priority: Literal["p0", "p1", "p2", "p3"]
    customer_sentiment: Literal["calm", "frustrated", "angry"]
    repro_steps: List[str]
    expected_behavior: str
    actual_behavior: str
    suggested_next_action: str

    @field_validator("category", mode="before")
    @classmethod
    def normalize_category(cls, v: str) -> str:
        return CATEGORY_MAP.get(str(v).strip().lower(), "other")

    @field_validator("priority", mode="before")
    @classmethod
    def normalize_priority(cls, v: str) -> str:
        mapped = PRIORITY_MAP.get(str(v).strip().lower())
        if mapped:
            return mapped
        raise ValueError(f"Cannot map priority '{v}' to p0/p1/p2/p3. Valid inputs: {list(PRIORITY_MAP.keys())}")

    @field_validator("customer_sentiment", mode="before")
    @classmethod
    def normalize_sentiment(cls, v: str) -> str:
        return SENTIMENT_MAP.get(str(v).strip().lower(), "frustrated")

## Instructor Client Setup

We use [Instructor](https://github.com/instructor-ai/instructor) to wrap the OpenAI
client, giving us Pydantic-validated structured output with automatic retries. When the
model returns a value that our validators can't normalize (like `"priority": "ASAP"`),
the validator raises, Instructor injects the error into the prompt, and the model gets
another chance with explicit feedback about what went wrong.

This is **Tier 2** — stronger than prompt engineering (Tier 3), more universal than strict
schema mode (Tier 1). It works with any model that can produce JSON-ish output.

In [5]:
instructor_client = instructor.from_openai(
    OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    ),
    mode=instructor.Mode.JSON,  # works with any model that can produce JSON
)

## The Customer Message

A single support ticket that any model should be able to triage. The customer is clearly
frustrated, has an urgent deadline (demo in one hour), and has already tried self-service
(reset password twice). Let's see how different models read the same situation.

In [6]:
PROMPT = """Turn this into a structured support ticket.

Customer message:
"I cannot log in since yesterday. I reset my password twice and it still says 'invalid credentials'.
I have a demo in one hour. This is ridiculous. Please fix this now."
"""

## API Helper

`call_openrouter` handles retries on 5xx errors with jittered backoff.
Raises on 4xx (bad request, auth failure) immediately. Used by the strict mode
and degraded mode paths (Tiers 1 and 3). Tier 2 uses the Instructor client instead.

In [7]:
def call_openrouter(payload: Dict[str, Any], retries: int = 2, timeout_s: int = 25) -> Dict[str, Any]:
    last = None
    for _ in range(retries):
        r = requests.post(
            f"{BASE}/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
            },
            json=payload,
            timeout=timeout_s,
        )

        if 500 <= r.status_code <= 599:
            last = f"{r.status_code}: {r.text[:240]}"
            time.sleep(0.4 + random.random() * 0.6)
            continue

        if r.status_code >= 400:
            raise RuntimeError(f"{r.status_code}: {r.text[:600]}")

        return r.json()

    raise RuntimeError(f"OpenRouter failed after retries. Last={last}")

## The Three-Tier Fallback Strategy

### Tier 1: Strict JSON Schema Mode (`response_format=json_schema`)

This is the strongest option available. We pass the full JSON schema directly to the API
with `"strict": True` and let the provider enforce it at the token-generation level.

But be aware that strict mode only constrains a **single LLM completion call** — the
model's immediate token generation is forced to match the schema. In a production agentic
system, you have different scenarios where this breaks down:

- **Tool calls happen in between.** The model generates a tool call, gets a result back,
  and then generates the final structured output. The tool result can inject unexpected
  content that shifts the model's "interpretation" of the schema values.
- **Multi-turn handoffs.** If the structured output from Model A feeds into a prompt for
  Model B (or even another turn of Model A), the schema enforcement only applies at each
  individual generation boundary, not across the chain.
- **Provider-level quirks.** OpenRouter is proxying to underlying providers. The "strict"
  enforcement depends on the provider actually implementing constrained decoding, not just
  claiming to.
- **The values still drift.** Even with strict mode working, your fallback model may
  interpret the same ticket differently — assigning different priorities, reading
  different sentiments.

So strict mode gives you well-formed JSON for one call. It does **not** give you
consistent *judgment* across models, across turns, or across your pipeline.

### Tier 2: Instructor + Pydantic Validators

When strict mode fails, we use Instructor with our Pydantic model. Instructor asks the
model for JSON, validates the response against `SupportTicket` (including the
`field_validator`s that normalize values), and **retries with the validation error
injected into the prompt** if anything fails. This is strictly more powerful than prompt
engineering because the model gets explicit feedback about what went wrong.

### Tier 3: Prompt-Only + Canonicalization (Last Resort)

When even Instructor can't get valid output — maybe the model doesn't support JSON mode
at all, or it fails all retries — we fall back to plain prompt engineering. We describe
the schema constraints in the system message and parse whatever comes back. Then
`canonicalize()` normalizes keys, maps values, and fills defaults. This is the weakest
path, but it's better than crashing.

In [8]:
STRICT_SYSTEM = "Return ONLY valid JSON. No markdown. No code fences."

def run_strict(model: str) -> Dict[str, Any]:
    """Tier 1: Strict JSON Schema mode via the provider."""
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": STRICT_SYSTEM},
            {"role": "user", "content": PROMPT},
        ],
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "support_ticket", "strict": True, "schema": SUPPORT_TICKET_SCHEMA},
        },
    }
    t0 = time.time()
    data = call_openrouter(payload)
    latency = time.time() - t0
    content = data["choices"][0]["message"]["content"]
    return {"model": model, "mode": "strict", "latency_s": round(latency, 3), "raw_content": content}

### Tier 2: Instructor-Powered Structured Output

This is where we move from hoping the model cooperates to **enforcing** it. Instructor
sends the Pydantic schema as part of the request, parses the response, runs the
`field_validator`s (which normalize values via our maps), and if anything fails, retries
with the validation error injected into the conversation.

`max_retries=2` means up to 3 total attempts (initial + 2 retries). If the model
consistently fails, we fall through to Tier 3.

In [9]:
def run_instructor(model: str) -> Dict[str, Any]:
    """Tier 2: Instructor + Pydantic validation with automatic retries."""
    t0 = time.time()
    ticket = instructor_client.chat.completions.create(
        model=model,
        response_model=SupportTicket,
        max_retries=2,
        messages=[
            {"role": "system", "content": "You are a customer support triage agent. Return a structured support ticket."},
            {"role": "user", "content": PROMPT},
        ],
    )
    latency = time.time() - t0
    return {
        "model": model,
        "mode": "instructor",
        "latency_s": round(latency, 3),
        "ticket": ticket,
        "raw_content": ticket.model_dump_json(indent=2),
    }

### Tier 3: Prompt-Only JSON (Last Resort)

When even Instructor can't get valid output — maybe the model doesn't support JSON mode
at all, or it exhausts all retries — we fall back to **prompt engineering**. We describe
the schema constraints in the system message and ask the model to return plain JSON.

This is inherently the least reliable path. Models may:
- Wrap JSON in \`\`\`json code fences
- Nest objects instead of returning a flat structure
- Use different key names (`subject` instead of `summary`)
- Pick values outside the allowed enums (`"High"` instead of `"p1"`)

That's why this path feeds into `canonicalize()` — the emergency normalizer.

In [10]:
DEGRADED_SYSTEM = """Return ONLY valid JSON. No markdown. No code fences.
Return a SINGLE flat JSON object with exactly these keys:
ticket_id, summary, category, priority, customer_sentiment, repro_steps, expected_behavior, actual_behavior, suggested_next_action.

Rules:
- category must be one of: billing, login, bug, feature_request, performance, other
- priority must be one of: p0, p1, p2, p3
- customer_sentiment must be one of: calm, frustrated, angry
- repro_steps must be a JSON array of strings (even if empty)
- Do not add any other keys
"""

def run_degraded(model: str) -> Dict[str, Any]:
    """Tier 3: Prompt-only JSON — last resort."""
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": DEGRADED_SYSTEM},
            {"role": "user", "content": PROMPT},
        ],
    }
    t0 = time.time()
    data = call_openrouter(payload)
    latency = time.time() - t0
    content = data["choices"][0]["message"]["content"]
    return {"model": model, "mode": "degraded", "latency_s": round(latency, 3), "raw_content": content}

## Robust JSON Parsing (for Tier 3)

Models in degraded mode often wrap their JSON in \`\`\`json code fences, add commentary
before or after the JSON, or return nested objects. We need to handle all of these
gracefully — because in production, you don't get to re-prompt.

In [11]:
def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\n?", "", s)
        s = re.sub(r"\n?```$", "", s)
    return s.strip()

def extract_first_json_object(text: str) -> str:
    s = text.strip()
    start = s.find("{")
    if start == -1:
        raise ValueError("No JSON object start found")
    depth = 0
    for i in range(start, len(s)):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                return s[start:i+1]
    raise ValueError("No complete JSON object found")

def parse_json_robust(content: str) -> Dict[str, Any]:
    cleaned = strip_code_fences(content)
    try:
        return json.loads(cleaned)
    except Exception:
        return json.loads(extract_first_json_object(cleaned))

## Last-Resort Canonicalization (for Tier 3)

The `canonicalize()` function is the emergency room for model output. It only runs when
both Tier 1 (strict) and Tier 2 (Instructor) have failed, which means the output is
probably messy. It:

- Accepts alternate key names (`subject` → `summary`, `steps_taken` → `repro_steps`)
- Maps non-standard values using our normalization maps
- Infers priority from context when all else fails
- Provides sensible defaults for missing fields

Note that the normalization maps are already defined above and are *also* used by the
Pydantic validators in Tier 2. This function is the backstop for when even the validators
couldn't help — typically because the model returned something so far off-schema that
Instructor's retries couldn't fix it.

In [12]:
def to_list(x: Any) -> List[str]:
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if "\n" in s:
            parts = [p.strip(" -\t") for p in s.split("\n")]
            return [p for p in parts if p]
        return [s]
    return []

def infer_priority(original_prompt: str, summary: str, sentiment: str, category: str) -> str:
    """Heuristic priority when the model gave us nothing usable."""
    t = (original_prompt + " " + (summary or "")).lower()
    s = (sentiment or "").lower()
    cat = (category or "").lower()

    urgent_signal = ("demo" in t and ("1 hour" in t or "one hour" in t)) or "urgent" in t or "immediate" in t
    blocked_login = (cat == "login") and ("cannot log in" in t or "unable to log in" in t)

    if blocked_login and ("angry" in s or urgent_signal):
        return "p0"
    if urgent_signal:
        return "p1"
    if blocked_login:
        return "p1"
    return "p2"

def canonicalize(raw: Dict[str, Any]) -> Dict[str, Any]:
    """Emergency normalizer for Tier 3 output. Maps alternate keys and values."""
    ticket_id = str(raw.get("ticket_id") or raw.get("id") or "TCK-0000").strip()
    summary = str(raw.get("summary") or raw.get("subject") or raw.get("title") or "").strip()

    category_raw = str(raw.get("category") or "other").strip().lower()
    category = CATEGORY_MAP.get(category_raw, "other")

    sentiment_raw = str(raw.get("customer_sentiment") or raw.get("sentiment") or "frustrated").strip().lower()
    sentiment = SENTIMENT_MAP.get(sentiment_raw, "frustrated")

    priority_raw = str(raw.get("priority") or "").strip().lower()
    priority = PRIORITY_MAP.get(priority_raw, None)
    if priority is None:
        priority = infer_priority(PROMPT, summary, sentiment, category)

    repro_steps = to_list(raw.get("repro_steps") or raw.get("steps_taken") or raw.get("actions_taken"))
    expected_behavior = str(raw.get("expected_behavior") or "").strip()
    actual_behavior = str(raw.get("actual_behavior") or raw.get("current_behavior") or "").strip()
    next_action = str(raw.get("suggested_next_action") or raw.get("requested_action") or "").strip()

    return {
        "ticket_id": ticket_id if ticket_id else "TCK-0000",
        "summary": summary if summary else "Customer cannot access account",
        "category": category,
        "priority": priority,
        "customer_sentiment": sentiment,
        "repro_steps": repro_steps if repro_steps else ["Attempt login", "Reset password", "Attempt login again"],
        "expected_behavior": expected_behavior if expected_behavior else "User should be able to log in successfully.",
        "actual_behavior": actual_behavior if actual_behavior else "User receives invalid credentials error after password reset.",
        "suggested_next_action": next_action if next_action else "Check auth logs and account lockout status; escalate if needed.",
    }

## The Full Three-Tier Pipeline

`run_model()` implements the complete strategy:

1. **Tier 1 — Try strict mode** → parse → validate with Pydantic (validators normalize)
2. **Tier 2 — Try Instructor** → Pydantic validates + normalizes → retries on failure
3. **Tier 3 — Prompt-only** → robust parse → `canonicalize()` → validate with Pydantic

Each tier is progressively less reliable but more universally compatible. In production,
most calls should succeed at Tier 1. If your primary model goes down and the fallback
can't do strict mode, Tier 2 catches it. If even that fails, Tier 3 is the emergency path.

The question isn't *if* your primary model will fail — it's *when*, and whether your
system degrades gracefully or crashes.

In [13]:
def run_model(model: str) -> Dict[str, Any]:
    strict_failure = None
    instructor_failure = None

    # ── Tier 1: Strict JSON Schema mode ──────────────────────────────
    try:
        out = run_strict(model)
        raw = parse_json_robust(out["raw_content"])
        ticket = SupportTicket.model_validate(raw)

        return {
            "model": model,
            "mode": "strict",
            "tier": 1,
            "strict_failed": False,
            "instructor_failed": False,
            "ok": True,
            "latency_s": out["latency_s"],
            "priority": ticket.priority,
            "category": ticket.category,
            "sentiment": ticket.customer_sentiment,
            "n_repro_steps": len(ticket.repro_steps),
            "strict_failure": None,
            "instructor_failure": None,
            "note": "Tier 1: Strict JSON Schema succeeded.",
            "raw_excerpt": out["raw_content"][:300],
        }
    except Exception as e_strict:
        strict_failure = f"{type(e_strict).__name__}: {str(e_strict)[:220]}"

    # ── Tier 2: Instructor + Pydantic validation ─────────────────────
    try:
        out2 = run_instructor(model)
        ticket2 = out2["ticket"]

        return {
            "model": model,
            "mode": "instructor",
            "tier": 2,
            "strict_failed": True,
            "instructor_failed": False,
            "ok": True,
            "latency_s": out2["latency_s"],
            "priority": ticket2.priority,
            "category": ticket2.category,
            "sentiment": ticket2.customer_sentiment,
            "n_repro_steps": len(ticket2.repro_steps),
            "strict_failure": strict_failure,
            "instructor_failure": None,
            "note": "Tier 2: Strict failed \u2192 Instructor + Pydantic validators succeeded.",
            "raw_excerpt": out2["raw_content"][:300],
        }
    except Exception as e_instr:
        instructor_failure = f"{type(e_instr).__name__}: {str(e_instr)[:220]}"

    # ── Tier 3: Prompt-only + canonicalize ───────────────────────────
    try:
        out3 = run_degraded(model)
        raw3 = parse_json_robust(out3["raw_content"])
        canonical = canonicalize(raw3)
        ticket3 = SupportTicket.model_validate(canonical)

        return {
            "model": model,
            "mode": "degraded_canonicalized",
            "tier": 3,
            "strict_failed": True,
            "instructor_failed": True,
            "ok": True,
            "latency_s": out3["latency_s"],
            "priority": ticket3.priority,
            "category": ticket3.category,
            "sentiment": ticket3.customer_sentiment,
            "n_repro_steps": len(ticket3.repro_steps),
            "strict_failure": strict_failure,
            "instructor_failure": instructor_failure,
            "note": "Tier 3: Strict + Instructor failed \u2192 prompt-only + canonicalize \u2192 validated.",
            "raw_excerpt": out3["raw_content"][:300],
        }
    except Exception as e_degraded:
        return {
            "model": model,
            "mode": "all_failed",
            "tier": 0,
            "strict_failed": True,
            "instructor_failed": True,
            "ok": False,
            "latency_s": 0,
            "priority": "unknown",
            "category": "unknown",
            "sentiment": "unknown",
            "n_repro_steps": 0,
            "strict_failure": strict_failure,
            "instructor_failure": instructor_failure,
            "note": f"All tiers failed. Last error: {str(e_degraded)[:200]}",
            "raw_excerpt": "",
        }

## Run All 4 Models × 2 Trials

We run each model **twice** on the same input to check:
- **Cross-model drift**: Do different models produce different priorities/sentiments?
- **Run-to-run stability**: Does the same model give the same answer twice?
- **Tier distribution**: Which models succeed at Tier 1 vs. needing Tier 2 or 3?

In [14]:
MODELS = [
    "openai/gpt-5.2",
    "qwen/qwen3-max-thinking",
    "anthropic/claude-haiku-4.5",
    "minimax/minimax-m2.5",
]

results = []
for m in MODELS:
    for trial in range(2):
        print(f"  Running {m} (trial {trial+1}/2)...")
        results.append(run_model(m))

print(f"\nDone: {len(results)} runs completed.")

  Running openai/gpt-5.2 (trial 1/2)...
  Running openai/gpt-5.2 (trial 2/2)...
  Running qwen/qwen3-max-thinking (trial 1/2)...
  Running qwen/qwen3-max-thinking (trial 2/2)...
  Running anthropic/claude-haiku-4.5 (trial 1/2)...
  Running anthropic/claude-haiku-4.5 (trial 2/2)...
  Running minimax/minimax-m2.5 (trial 1/2)...
  Running minimax/minimax-m2.5 (trial 2/2)...

Done: 8 runs completed.


## Side-by-Side Comparison: What Did Each Model Produce?

This is where it gets real. Pay attention to:
- **Tier**: Which fallback level each model landed on — this tells you what broke
- **Priority**: `p0` vs `p1` — determines whether someone gets paged at 2am or the ticket waits until morning
- **Sentiment**: `angry` vs `frustrated` — changes the tone of the auto-response and the escalation path
- **Repro steps count**: how much detail the model extracted from the same message
- **Latency**: the real cost of falling through to lower tiers

In [15]:
# ── Per-run detail table ──────────────────────────────────────────────
print(f"{'Model':<30} {'#':>2} {'Tier':>4} {'Mode':<25} {'Pri':>3} {'Sentiment':<12} {'Steps':>5} {'Latency':>8}")
print("\u2500" * 95)
for i, r in enumerate(results):
    trial = (i % 2) + 1
    print(f"{r['model']:<30} {trial:>2} {r['tier']:>4} {r['mode']:<25} {r['priority']:>3} {r['sentiment']:<12} {r['n_repro_steps']:>5} {r['latency_s']:>7.1f}s")

# ── Highlight cross-model disagreements ───────────────────────────────
priorities = {}
sentiments = {}
tiers = {}
for r in results:
    short = r["model"].split("/")[-1]
    priorities.setdefault(short, set()).add(r["priority"])
    sentiments.setdefault(short, set()).add(r["sentiment"])
    tiers.setdefault(short, set()).add(r["tier"])

print()
print("=" * 70)
print(" CROSS-MODEL DRIFT: Same customer, same schema, different judgments")
print("=" * 70)

print()
print("Priority assignments (p0 = page someone, p1 = next morning):")
for m, ps in priorities.items():
    marker = "  \u2713" if len(ps) == 1 else "  \u26a0 UNSTABLE"
    print(f"  {m:<28} \u2192 {', '.join(sorted(ps))}{marker}")

print()
print("Sentiment readings (angry = escalate, frustrated = empathize):")
for m, ss in sentiments.items():
    marker = "  \u2713" if len(ss) == 1 else "  \u26a0 UNSTABLE"
    print(f"  {m:<28} \u2192 {', '.join(sorted(ss))}{marker}")

print()
print("Tier distribution (which fallback level did each model land on?):")
for m, ts in tiers.items():
    tier_str = ", ".join(f"Tier {t}" for t in sorted(ts))
    print(f"  {m:<28} \u2192 {tier_str}")

# ── Strict + Instructor failures ─────────────────────────────────────
strict_failures = [r for r in results if r["strict_failed"]]
instructor_failures = [r for r in results if r.get("instructor_failed")]
if strict_failures:
    print()
    print("=" * 70)
    print(f" TIER 1 FAILURES: {len(strict_failures)} of {len(results)} runs failed strict mode")
    print("=" * 70)
    for r in strict_failures:
        print(f"  {r['model']}")
        print(f"    Strict error: {(r['strict_failure'] or '')[:120]}")
        if r.get("instructor_failed"):
            print(f"    Instructor error: {(r.get('instructor_failure') or '')[:120]}")
            print(f"    Rescued by: Tier 3 (prompt-only + canonicalize)")
        else:
            print(f"    Rescued by: Tier 2 (Instructor + Pydantic)")

# ── Latency comparison by tier ────────────────────────────────────────
print()
print("=" * 70)
print(" LATENCY BY TIER")
print("=" * 70)
for tier_num, tier_name in [(1, "Strict"), (2, "Instructor"), (3, "Degraded+canonicalize")]:
    lats = [r["latency_s"] for r in results if r["tier"] == tier_num]
    if lats:
        print(f"  Tier {tier_num} ({tier_name}): avg {sum(lats)/len(lats):.1f}s  (n={len(lats)})")

Model                           # Tier Mode                      Pri Sentiment    Steps  Latency
───────────────────────────────────────────────────────────────────────────────────────────────
openai/gpt-5.2                  1    1 strict                     p0 angry            7     6.2s
openai/gpt-5.2                  2    1 strict                     p0 angry            7     6.9s
qwen/qwen3-max-thinking         1    1 strict                     p1 frustrated       4     6.2s
qwen/qwen3-max-thinking         2    1 strict                     p1 frustrated       4     6.0s
anthropic/claude-haiku-4.5      1    1 strict                     p0 angry            7     7.0s
anthropic/claude-haiku-4.5      2    1 strict                     p0 angry            5     5.8s
minimax/minimax-m2.5            1    1 strict                     p1 frustrated       6     5.5s
minimax/minimax-m2.5            2    1 strict                     p0 frustrated       3     2.6s

 CROSS-MODEL DRIFT: Same custo

## Raw Output Samples

Let's look at what the models actually returned. This makes the drift concrete —
you can see exactly how different models format the same information differently,
and which tier caught them.

In [16]:
seen_models = []
for r in results:
    if r["model"] not in seen_models:
        seen_models.append(r["model"])

for model_name in seen_models:
    model_runs = [r for r in results if r["model"] == model_name]
    short = model_name.split("/")[-1]
    r0 = model_runs[0]
    print(f"{'\u2501' * 80}")
    print(f"  {short}  |  tier: {r0['tier']}  |  mode: {r0['mode']}  |  priority: {r0['priority']}  |  sentiment: {r0['sentiment']}")
    print(f"{'\u2501' * 80}")

    # Show key fields from the raw output
    raw = r0["raw_excerpt"]
    try:
        # Try to parse (close truncated JSON if needed)
        parsed = json.loads(raw + ("}" * max(0, raw.count("{") - raw.count("}"))))
    except Exception:
        parsed = None

    if parsed:
        for key in ["ticket_id", "summary", "category", "priority", "customer_sentiment", "repro_steps"]:
            if key in parsed:
                val = parsed[key]
                if isinstance(val, list):
                    print(f"  {key}: [{len(val)} items] {', '.join(str(v)[:40] for v in val[:3])}{'...' if len(val) > 3 else ''}")
                else:
                    print(f"  {key}: {val}")
    else:
        print(f"  {raw[:300]}")

    # Show if the two trials agreed
    if len(model_runs) > 1:
        r1 = model_runs[1]
        diffs = []
        for field in ["priority", "sentiment", "n_repro_steps", "tier"]:
            if r0.get(field) != r1.get(field):
                diffs.append(f"{field}: {r0.get(field)} vs {r1.get(field)}")
        if diffs:
            print(f"  \u26a0 Trial 1 vs 2 differ: {'; '.join(diffs)}")
        else:
            print(f"  \u2713 Both trials agree on priority, sentiment, steps, tier")
    print()

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  gpt-5.2  |  tier: 1  |  mode: strict  |  priority: p0  |  sentiment: angry
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  {"ticket_id":"TCK-20260326-0001","summary":"User cannot log in; password reset twice but still gets 'invalid credentials' error; urgent due to demo in 1 hour","category":"login","priority":"p0","customer_sentiment":"angry","repro_steps":["Go to login page","Enter username/email and password","Submit
  ✓ Both trials agree on priority, sentiment, steps, tier

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  qwen3-max-thinking  |  tier: 1  |  mode: strict  |  priority: p1  |  sentiment: frustrated
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  {
  "ticket_id": "TKT-20231005-001",
  "summary": "Unable to log in after multiple password resets",
  "category": "login",
  "priority": "p1"

## Quantified Drift Analysis

Let's put numbers on the drift. The **drift score** combines:
- **Priority drift** (1 point if a model assigns different priorities across trials)
- **Steps drift** (normalized: how much the repro step count varies)
- **Tier penalty** (0.0 for Tier 1, 0.25 for Tier 2, 0.5 for Tier 3)

A drift score of 0 = perfectly stable at Tier 1. Higher = more unpredictable.

In [17]:
import pandas as pd

df = pd.DataFrame(results).copy()

# Add trial index per model
df["trial"] = df.groupby("model").cumcount() + 1

# ── Per-run summary table ────────────────────────────────────────────
summary_cols = ["model", "trial", "tier", "mode", "latency_s",
                "priority", "category", "sentiment", "n_repro_steps", "note"]
print("Per-Run Results")
print("=" * 40)
display(df[summary_cols])

Per-Run Results


,model,trial,tier,mode,latency_s,priority,category,sentiment,n_repro_steps,note
0,openai/gpt-5.2,1,1,strict,6.193,p0,login,angry,7,Tier 1: Strict JSON Schema succeeded.
1,openai/gpt-5.2,2,1,strict,6.881,p0,login,angry,7,Tier 1: Strict JSON Schema succeeded.
2,qwen/qwen3-max-thinking,1,1,strict,6.182,p1,login,frustrated,4,Tier 1: Strict JSON Schema succeeded.
3,qwen/qwen3-max-thinking,2,1,strict,6.003,p1,login,frustrated,4,Tier 1: Strict JSON Schema succeeded.
4,anthropic/claude-haiku-4.5,1,1,strict,7.000,p0,login,angry,7,Tier 1: Strict JSON Schema succeeded.
5,anthropic/claude-haiku-4.5,2,1,strict,5.772,p0,login,angry,5,Tier 1: Strict JSON Schema succeeded.
6,minimax/minimax-m2.5,1,1,strict,5.471,p1,login,frustrated,6,Tier 1: Strict JSON Schema succeeded.
7,minimax/minimax-m2.5,2,1,strict,2.600,p0,login,frustrated,3,Tier 1: Strict JSON Schema succeeded.


In [18]:
# ── Per-model delta: what changed between trials? ─────────────────────
def delta_summary(g: pd.DataFrame) -> pd.Series:
    diffs = []
    for field in ["priority", "category", "sentiment", "n_repro_steps", "mode", "tier"]:
        if g[field].nunique(dropna=True) > 1:
            diffs.append(field)

    change = ("Changed across trials: " + ", ".join(diffs) + ".") if diffs else "Stable across both trials."

    return pd.Series({
        "trials": len(g),
        "changed_fields": ", ".join(diffs) if diffs else "none",
        "delta_note": change,
        "latency_min": round(float(g["latency_s"].min()), 3),
        "latency_max": round(float(g["latency_s"].max()), 3),
    })

delta = df.groupby("model").apply(delta_summary, include_groups=False).reset_index()
print("\nRun-to-Run Stability (same model, same input)")
print("=" * 50)
display(delta)

# ── Drift scoring (updated for three tiers) ──────────────────────────
priority_rank = {"p0": 3, "p1": 2, "p2": 1, "p3": 0}
tier_penalty_map = {1: 0.0, 2: 0.25, 3: 0.5, 0: 1.0}

def drift_for_model(g: pd.DataFrame) -> pd.Series:
    pr = g["priority"].map(priority_rank)
    priority_drift = int(pr.nunique(dropna=True) > 1)

    steps = pd.to_numeric(g["n_repro_steps"], errors="coerce")
    steps_min = steps.min() if steps.notna().any() else None
    steps_max = steps.max() if steps.notna().any() else None
    steps_drift = float(steps_max - steps_min) if steps_min is not None else 0.0
    steps_drift_norm = min(1.0, steps_drift / 10.0)

    # Tier penalty: worst tier used across trials
    worst_tier = int(g["tier"].max()) if g["tier"].notna().any() else 0
    tier_penalty = tier_penalty_map.get(worst_tier, 0.5)

    drift_score = priority_drift + steps_drift_norm + tier_penalty

    return pd.Series({
        "trials": len(g),
        "tiers_used": ", ".join(f"T{int(t)}" for t in sorted(g["tier"].dropna().unique())),
        "worst_tier": worst_tier,
        "priority_values": ", ".join(sorted(g["priority"].dropna().unique().tolist())),
        "steps_min": int(steps_min) if steps_min is not None else None,
        "steps_max": int(steps_max) if steps_max is not None else None,
        "priority_drift": priority_drift,
        "steps_drift": round(steps_drift, 2),
        "tier_penalty": tier_penalty,
        "drift_score": round(drift_score, 2),
        "avg_latency_s": round(float(g["latency_s"].mean()), 3),
        "p95_latency_s": round(float(g["latency_s"].quantile(0.95)), 3),
    })

drift = df.groupby("model").apply(drift_for_model, include_groups=False).reset_index()
print("\nDrift Score by Model (lower = more stable)")
print("=" * 50)
display(drift)


Run-to-Run Stability (same model, same input)


,model,trials,changed_fields,delta_note,latency_min,latency_max
0,anthropic/claude-haiku-4.5,2,n_repro_steps,Changed across trials: n_repro_steps.,5.772,7.000
1,minimax/minimax-m2.5,2,"priority, n_repro_steps","Changed across trials: priority, n_repro_steps.",2.600,5.471
2,openai/gpt-5.2,2,none,Stable across both trials.,6.193,6.881
3,qwen/qwen3-max-thinking,2,none,Stable across both trials.,6.003,6.182



Drift Score by Model (lower = more stable)


,model,trials,tiers_used,worst_tier,priority_values,steps_min,steps_max,priority_drift,steps_drift,tier_penalty,drift_score,avg_latency_s,p95_latency_s
0,anthropic/claude-haiku-4.5,2,T1,1,p0,5,7,0,2.0,0.0,0.2,6.386,6.939
1,minimax/minimax-m2.5,2,T1,1,"p0, p1",3,6,1,3.0,0.0,1.3,4.035,5.327
2,openai/gpt-5.2,2,T1,1,p0,7,7,0,0.0,0.0,0.0,6.537,6.847
3,qwen/qwen3-max-thinking,2,T1,1,p1,4,4,0,0.0,0.0,0.0,6.093,6.173


In [19]:
# ── Overall summary ──────────────────────────────────────────────────
tier_counts = df["tier"].value_counts().to_dict()
overall = {
    "models_tested": int(df["model"].nunique()),
    "total_runs": int(len(df)),
    "tier_1_runs": int(tier_counts.get(1, 0)),
    "tier_2_runs": int(tier_counts.get(2, 0)),
    "tier_3_runs": int(tier_counts.get(3, 0)),
    "all_failed_runs": int(tier_counts.get(0, 0)),
    "avg_latency_s": round(float(df["latency_s"].mean()), 3),
    "p95_latency_s": round(float(df["latency_s"].quantile(0.95)), 3),
    "priority_values_seen": sorted(df["priority"].dropna().unique().tolist()),
    "sentiment_values_seen": sorted(df["sentiment"].dropna().unique().tolist()),
    "steps_range": (
        int(df["n_repro_steps"].min()) if df["n_repro_steps"].notna().any() else None,
        int(df["n_repro_steps"].max()) if df["n_repro_steps"].notna().any() else None,
    ),
}

print("Overall Summary")
print("=" * 40)
for k, v in overall.items():
    print(f"  {k}: {v}")

# Highlight the key finding
pri_set = set(df["priority"].dropna().unique())
sent_set = set(df["sentiment"].dropna().unique())
if len(pri_set) > 1 or len(sent_set) > 1:
    print()
    print("\u26a0  KEY FINDING: Same customer message, same schema, but models disagree:")
    if len(pri_set) > 1:
        print(f"   Priority: {sorted(pri_set)} \u2192 This changes who gets paged and when.")
    if len(sent_set) > 1:
        print(f"   Sentiment: {sorted(sent_set)} \u2192 This changes the auto-response tone and escalation path.")

Overall Summary
  models_tested: 4
  total_runs: 8
  tier_1_runs: 8
  tier_2_runs: 0
  tier_3_runs: 0
  all_failed_runs: 0
  avg_latency_s: 5.763
  p95_latency_s: 6.958
  priority_values_seen: ['p0', 'p1']
  sentiment_values_seen: ['angry', 'frustrated']
  steps_range: (3, 7)

⚠  KEY FINDING: Same customer message, same schema, but models disagree:
   Priority: ['p0', 'p1'] → This changes who gets paged and when.
   Sentiment: ['angry', 'frustrated'] → This changes the auto-response tone and escalation path.
